In [3]:
# segment_peers_list.py
# Step 1 - Define all segment peers and save as CSV

import pandas as pd

segment_peers = [

    # FMCG Cigarettes
    ("FMCG Cigarettes", "VST Industries Ltd",         "VSTIND.NS"),
    ("FMCG Cigarettes", "Godfrey Phillips India Ltd",  "GODFRYPHP.NS"),

    # FMCG Others
    ("FMCG Others",     "Hindustan Unilever Ltd",      "HINDUNILVR.NS"),
    ("FMCG Others",     "Dabur India Ltd",             "DABUR.NS"),
    ("FMCG Others",     "Marico Limited",              "MARICO.NS"),
    ("FMCG Others",     "Godrej Consumer Products",    "GODREJCP.NS"),
    ("FMCG Others",     "Britannia Industries Ltd",    "BRITANNIA.NS"),

    # Agri Business
    ("Agri Business",   "KRBL Limited",                "KRBL.NS"),
    ("Agri Business",   "LT Foods Limited",            "LTFOODS.NS"),
    ("Agri Business",   "Avanti Feeds Limited",        "AVANTIFEED.NS"),
    ("Agri Business",   "Godrej Agrovet Limited",      "GODREJAGRO.NS"),
    ("Agri Business",   "Coromandel International",    "COROMANDEL.NS"),

    # Paperboards
    ("Paperboards Paper Pack.", "JK Paper Limited",            "JKPAPER.NS"),
    ("Paperboards Paper Pack.", "Tamilnadu Newsprint & Paper", "TNPL.NS"),
    ("Paperboards Paper Pack.", "Century Plyboards (I) Ltd",   "CENTURYPLY.NS"),
    ("Paperboards Paper Pack.", "Emami Paper Mills Limited",   "EMAMIPAP.NS"),

    # Others / Infotech
    ("Others / Infotech", "Mphasis",           "MPHASIS.NS"),
    ("Others / Infotech", "Coforge",           "COFORGE.NS"),
    ("Others / Infotech", "Birlasoft",         "BSOFT.NS"),
    ("Others / Infotech", "KPIT Technologies", "KPITTECH.NS"),
]

df = pd.DataFrame(segment_peers, columns=["Segment", "Company", "Ticker"])
df.to_csv("segment_peers_list.csv", index=False)

print("Peer list saved!")
print(df.to_string(index=False))

Peer list saved!
                Segment                     Company        Ticker
        FMCG Cigarettes          VST Industries Ltd     VSTIND.NS
        FMCG Cigarettes  Godfrey Phillips India Ltd  GODFRYPHP.NS
            FMCG Others      Hindustan Unilever Ltd HINDUNILVR.NS
            FMCG Others             Dabur India Ltd      DABUR.NS
            FMCG Others              Marico Limited     MARICO.NS
            FMCG Others    Godrej Consumer Products   GODREJCP.NS
            FMCG Others    Britannia Industries Ltd  BRITANNIA.NS
          Agri Business                KRBL Limited       KRBL.NS
          Agri Business            LT Foods Limited    LTFOODS.NS
          Agri Business        Avanti Feeds Limited AVANTIFEED.NS
          Agri Business      Godrej Agrovet Limited GODREJAGRO.NS
          Agri Business    Coromandel International COROMANDEL.NS
Paperboards Paper Pack.            JK Paper Limited    JKPAPER.NS
Paperboards Paper Pack. Tamilnadu Newsprint & Paper       T

In [4]:
# fetch_segment_tickers.py
# Step 2 - Search each company name on Yahoo Finance and get the NSE ticker

import yfinance as yf
import pandas as pd

peers = pd.read_csv("segment_peers_list.csv")

tickers_found = []

for _, row in peers.iterrows():

    name    = row["Company"]
    segment = row["Segment"]

    print(f"Searching ticker for: {name}...")

    try:
        # Search Yahoo Finance by company name
        results = yf.Search(name, max_results=5).quotes

        # Pick the first NSE result
        ticker = None
        for result in results:
            symbol = result.get("symbol", "")
            if symbol.endswith(".NS"):
                ticker = symbol
                break

        # If no .NS found, just take the first result
        if not ticker and results:
            ticker = results[0].get("symbol", None)

    except Exception as e:
        print(f"  Error for {name}: {e}")
        ticker = None

    print(f"  Found: {ticker}")

    tickers_found.append({
        "Segment" : segment,
        "Company" : name,
        "Ticker"  : ticker,
    })

df = pd.DataFrame(tickers_found)
df.to_csv("segment_peers_with_tickers.csv", index=False)

print("\nTickers saved!")
print(df.to_string(index=False))

Searching ticker for: VST Industries Ltd...
  Found: VSTIND.NS
Searching ticker for: Godfrey Phillips India Ltd...
  Found: GODFRYPHLP.BO
Searching ticker for: Hindustan Unilever Ltd...
  Found: HINDUNILVR.NS
Searching ticker for: Dabur India Ltd...
  Found: DABUR.NS
Searching ticker for: Marico Limited...
  Found: MARICO.NS
Searching ticker for: Godrej Consumer Products...
  Found: GODREJCP.NS
Searching ticker for: Britannia Industries Ltd...
  Found: BRITANNIA.NS
Searching ticker for: KRBL Limited...
  Found: KRBL.NS
Searching ticker for: LT Foods Limited...
  Found: LTFOODS.NS
Searching ticker for: Avanti Feeds Limited...
  Found: AVANTIFEED.NS
Searching ticker for: Godrej Agrovet Limited...
  Found: GODREJAGRO.NS
Searching ticker for: Coromandel International...
  Found: COROMANDEL.NS
Searching ticker for: JK Paper Limited...
  Found: JKPAPER.NS
Searching ticker for: Tamilnadu Newsprint & Paper...
  Found: None
Searching ticker for: Century Plyboards (I) Ltd...
  Found: CENTURYPLY.

In [5]:
# fetch_segment_raw.py
# Step 3 - Use fetched tickers to pull Market Cap, Debt, Cash, EBIT

import yfinance as yf
import pandas as pd

peers = pd.read_csv("segment_peers_with_tickers.csv")

all_data = []

for _, row in peers.iterrows():

    name    = row["Company"]
    ticker  = row["Ticker"]
    segment = row["Segment"]

    print(f"Fetching financials: {name} ({ticker})...")

    try:
        stock = yf.Ticker(ticker)
        info  = stock.info
        bs    = stock.balance_sheet
        inc   = stock.financials

        # Market Cap
        mkt_cap    = info.get("marketCap", None)
        mkt_cap_cr = round(mkt_cap / 1e7, 2) if mkt_cap else None

        # Debt
        debt = None
        for label in ["Total Debt", "Long Term Debt", "Short Long Term Debt"]:
            if label in bs.index:
                val = bs.loc[label].iloc[0]
                if pd.notna(val):
                    debt = round(val / 1e7, 2)
                    break

        # Cash
        cash = None
        for label in ["Cash And Cash Equivalents", "Cash",
                      "Cash Cash Equivalents And Short Term Investments"]:
            if label in bs.index:
                val = bs.loc[label].iloc[0]
                if pd.notna(val):
                    cash = round(val / 1e7, 2)
                    break

        # EBIT
        ebit = None
        for label in ["EBIT", "Operating Income", "Ebit"]:
            if label in inc.index:
                val = inc.loc[label].iloc[0]
                if pd.notna(val):
                    ebit = round(val / 1e7, 2)
                    break

    except Exception as e:
        print(f"  Error for {name}: {e}")
        mkt_cap_cr = debt = cash = ebit = None

    all_data.append({
        "Segment"         : segment,
        "Company"         : name,
        "Ticker"          : ticker,
        "Market Cap (Cr)" : mkt_cap_cr,
        "Debt (Cr)"       : debt,
        "Cash (Cr)"       : cash,
        "EBIT (Cr)"       : ebit,
    })

df = pd.DataFrame(all_data)
df.to_csv("segment_raw.csv", index=False)

print("\nRaw financials saved!")
print(df.to_string(index=False))

Fetching financials: VST Industries Ltd (VSTIND.NS)...
Fetching financials: Godfrey Phillips India Ltd (GODFRYPHLP.BO)...
Fetching financials: Hindustan Unilever Ltd (HINDUNILVR.NS)...
Fetching financials: Dabur India Ltd (DABUR.NS)...
Fetching financials: Marico Limited (MARICO.NS)...
Fetching financials: Godrej Consumer Products (GODREJCP.NS)...
Fetching financials: Britannia Industries Ltd (BRITANNIA.NS)...
Fetching financials: KRBL Limited (KRBL.NS)...
Fetching financials: LT Foods Limited (LTFOODS.NS)...
Fetching financials: Avanti Feeds Limited (AVANTIFEED.NS)...
Fetching financials: Godrej Agrovet Limited (GODREJAGRO.NS)...
Fetching financials: Coromandel International (COROMANDEL.NS)...
Fetching financials: JK Paper Limited (JKPAPER.NS)...
Fetching financials: Tamilnadu Newsprint & Paper (nan)...
  Error for Tamilnadu Newsprint & Paper: 'float' object has no attribute 'upper'
Fetching financials: Century Plyboards (I) Ltd (CENTURYPLY.NS)...
Fetching financials: Emami Paper Mill

In [6]:
# calc_ev_ebit_final.py
# Step 4 - Calculate EV and EV/EBIT, save final output

import pandas as pd

df = pd.read_csv("segment_raw.csv")

# Fill missing debt and cash with 0
df["Debt (Cr)"] = df["Debt (Cr)"].fillna(0)
df["Cash (Cr)"] = df["Cash (Cr)"].fillna(0)

# EV = Market Cap + Debt - Cash
df["EV (Cr)"] = (df["Market Cap (Cr)"] + df["Debt (Cr)"] - df["Cash (Cr)"]).round(2)

# EV / EBIT
df["EV/EBIT"] = (df["EV (Cr)"] / df["EBIT (Cr)"]).round(2)

# Final column order
df = df[[
    "Segment",
    "Company",
    "Ticker",
    "Market Cap (Cr)",
    "Debt (Cr)",
    "Cash (Cr)",
    "EV (Cr)",
    "EBIT (Cr)",
    "EV/EBIT"
]]

df.to_csv("segment_peer_ev_ebit.csv", index=False)

print("Final output saved!")
print(df.to_string(index=False))

Final output saved!
                Segment                     Company        Ticker  Market Cap (Cr)  Debt (Cr)  Cash (Cr)   EV (Cr)  EBIT (Cr)  EV/EBIT
        FMCG Cigarettes          VST Industries Ltd     VSTIND.NS          4387.62       0.00      23.69   4363.93     348.74    12.51
        FMCG Cigarettes  Godfrey Phillips India Ltd GODFRYPHLP.BO         35251.88     177.49      14.25  35415.12    1483.36    23.87
            FMCG Others      Hindustan Unilever Ltd HINDUNILVR.NS        529597.88    1648.00    6071.00 525174.88   14810.00    35.46
            FMCG Others             Dabur India Ltd      DABUR.NS         78565.61     950.08     184.17  79331.52    2404.78    32.99
            FMCG Others              Marico Limited     MARICO.NS        100568.17     554.00     320.00 100802.17    2151.00    46.86
            FMCG Others    Godrej Consumer Products   GODREJCP.NS        109476.94    4004.49     454.92 113026.51    2983.00    37.89
            FMCG Others    Britanni